In [1]:
import pandas as pd
df = pd.read_csv("data/train.csv")
print(f"Total rows: {len(df)}")
print(df["toxic"].value_counts())  # Distribution
print(df.head(2))

Total rows: 159571
toxic
0    144277
1     15294
Name: count, dtype: int64
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  


In [36]:
import pandas as pd
df = pd.read_csv("data/adversarial_train_full_ALL.csv")
print(f"Total rows: {len(df)}")
# print(df["toxic"].value_counts())  # Distribution
print(df.columns)

Total rows: 1048575
Index(['original_text', 'adversarial_text', 'toxicity', 'label'], dtype='object')


In [27]:
import pandas as pd
df = pd.read_csv("data/fp_filter_results_step8.csv")
print(f"Total rows: {len(df)}")

print(df.columns)

Total rows: 654
Index(['text', 'predicted_label', 'confidence', 'important_tokens', 'flag'], dtype='object')


In [28]:
print(df["predicted_label"].value_counts())  # Distribution

predicted_label
0    575
1     79
Name: count, dtype: int64


In [23]:
import pandas as pd
df = pd.read_csv("data/test.csv")
print(f"Total rows: {len(df)}")

print(df.columns)

Total rows: 153164
Index(['id', 'comment_text'], dtype='object')


In [40]:
clean_df = pd.read_csv("data/test.csv")
ambiguous_df = pd.read_csv("data/fp_filter_results_step8.csv")
adversarial_df = pd.read_csv("data/adversarial_train_full_ALL.csv")

In [48]:
# ✅ Select 1000 rows from each (ensure enough data)
clean_subset = clean_df[['id', 'comment_text']].dropna().sample(n=1000, random_state=42, replace=False)

ambig_subset = ambiguous_df[['text']].dropna().sample(n=500, random_state=42, replace=False)
ambig_subset = ambig_subset.rename(columns={'text': 'comment_text'})
ambig_subset['id'] = ['ambig_' + str(i) for i in range(len(ambig_subset))]  # Generate fake IDs

adv_subset = adversarial_df[['adversarial_text']].dropna().sample(n=1000, random_state=42, replace=False)
adv_subset = adv_subset.rename(columns={'adversarial_text': 'comment_text'})
adv_subset['id'] = ['adv_' + str(i) for i in range(len(adv_subset))]

In [50]:
# ✅ Combine and shuffle
from sklearn.utils import shuffle


combined_df = pd.concat([clean_subset, ambig_subset, adv_subset], ignore_index=True)
combined_df = shuffle(combined_df, random_state=42).reset_index(drop=True)

# ✅ Save final dataset
output_path = "data/test_combined.csv"
combined_df.to_csv(output_path, index=False)

print(f"✅ Combined evaluation set saved to: {output_path} with shape {combined_df.shape}")

✅ Combined evaluation set saved to: data/test_combined.csv with shape (2500, 2)


In [56]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from tqdm import tqdm


final_df = pd.read_csv("data/test_combined.csv")
print(f"Total rows: {len(df)}")

print(df.columns)

Total rows: 2500
Index(['id', 'comment_text'], dtype='object')


In [57]:


# ✅ Load model + tokenizer
model_path = "checkpoints/best_model_step7_2.pkl"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model = torch.load(model_path, map_location=device)
model.to(device)
model.eval()



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [62]:

# ✅ Initialize lists
predictions = []
confidences = []

print("🔍 Running predictions on combined test set...")

for text in tqdm(final_df["comment_text"].astype(str).tolist()):
    try:
        # Tokenize and predict
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=1).squeeze()
            pred_label = torch.argmax(probs).item()
            confidence = probs[pred_label].item()

        predictions.append( 1 if pred_label == 1 else 0)
        confidences.append(round(confidence, 4))

    except Exception as e:
        predictions.append("ERROR")
        confidences.append(0)
        print(f"❌ Error: {e}")

# ✅ Add columns to DataFrame
final_df["prediction"] = predictions
final_df["confidence"] = confidences

# ✅ Save results
output_path = "data/test_combined_predictions.csv"
final_df.to_csv(output_path, index=False)
print(f"Saved predictions to: {output_path}")

🔍 Running predictions on combined test set...


100%|██████████| 2500/2500 [01:25<00:00, 29.09it/s]

Saved predictions to: data/test_combined_predictions.csv


In [63]:
import pandas as pd
df = pd.read_csv("data/test_combined_predictions.csv")
print(f"Total rows: {len(df)}")

print(df.columns)

Total rows: 2500
Index(['id', 'comment_text', 'prediction', 'confidence'], dtype='object')


In [64]:
print(df["prediction"].value_counts())  # Distribution

prediction
0    1255
1    1245
Name: count, dtype: int64


In [65]:
df

,id,comment_text,prediction,confidence
0,ambig_447,"Goodbye to a decent, funny and kind man, someo...",0,0.9997
1,ambig_114,Guess the powers that be can clearly see your ...,0,0.9973
2,ambig_64,"""\n Thanks for the pointer. Both the pages (:...",0,1.0000
3,adv_787,We are getting a taste of just what every id!o...,1,1.0000
4,adv_37,"Bran Cereal, milk, eggs, & Soy have B12",0,0.9998
...,...,...,...,...
2495,adv_138,I wonder what the outcry would have been if a ...,1,1.0000
2496,ambig_95,"Chase Gay\nHello, in relation to ur Question o...",0,0.9989
2497,ambig_130,Hello again Rodent\nI'm back again. Your a big...,0,0.9239
2498,ambig_294,"LOL, congrats on your google skills in digging...",1,0.9992
